### Generate the synthetic dataset

In [1]:
import numpy as np, pandas as pd
np.random.seed(42)
n = 2000
df = pd.DataFrame({
  'transaction_id': [f'TXN{i:05d}' for i in range(1, n+1)],
  'date': pd.date_range('2024-01-01', periods=n, freq='6h'),
  'category': np.random.choice(['Electronics','Clothing','Home & Garden','Books','Sports'], n),
  'region': np.random.choice(['North','South','East','West','Central'], n),
  'quantity': np.random.randint(1, 10, n),
  'unit_price': np.round(np.random.uniform(5, 500, n), 2),
  'payment_type': np.random.choice(['Credit Card','Debit Card','PayPal','Cash'], n),
  'customer_age': np.random.randint(18, 75, n),
})
df['total_value'] = np.round(df['quantity'] * df['unit_price'], 2)
missing_idx = np.random.choice(df.index, size=80, replace=False)
df.loc[missing_idx[:40], 'customer_age'] = np.nan
df.loc[missing_idx[40:], 'payment_type'] = np.nan

### Always inspect first

In [2]:
print(df.shape)
print(df.info())
print(df.describe())
print(df.isnull().sum())

(2000, 9)
<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   transaction_id  2000 non-null   str           
 1   date            2000 non-null   datetime64[us]
 2   category        2000 non-null   str           
 3   region          2000 non-null   str           
 4   quantity        2000 non-null   int32         
 5   unit_price      2000 non-null   float64       
 6   payment_type    1960 non-null   str           
 7   customer_age    1960 non-null   float64       
 8   total_value     2000 non-null   float64       
dtypes: datetime64[us](1), float64(3), int32(1), str(4)
memory usage: 132.9 KB
None
                      date     quantity   unit_price  customer_age  \
count                 2000  2000.000000  2000.000000   1960.000000   
mean   2024-09-06 21:00:00     5.021500   249.524350     45.860204   
min    2024-01-01 00:00:00   

### .loc vs .iloc — most common beginner confusion

In [3]:

high_value_txns = df[df['total_value'] > 1000]
elec_north = df[(df['category'] == 'Electronics') & (df['region'] == 'North')]
first_row = df.loc[0, ['transaction_id', 'category', 'total_value']]
last_5 = df.iloc[-5:, :4]

### groupby — the workhorse of data analysis

In [4]:
category_revenue = df.groupby('category')['total_value'].sum().sort_values(ascending=False)
avg_by_region = df.groupby('region')['total_value'].mean().round(2)
pivot = df.groupby(['category', 'region'])['total_value'].agg(['count', 'sum', 'mean'])
pivot.columns = ['num_transactions', 'total_revenue', 'avg_value']
pivot = pivot.reset_index()

### Reduce memory with dtype conversion

In [5]:
df_opt = df.copy()
df_opt['category'] = df_opt['category'].astype('category')
df_opt['region'] = df_opt['region'].astype('category')
df_opt['payment_type'] = df_opt['payment_type'].astype('category')
df_opt['quantity'] = df_opt['quantity'].astype('int8')
df_opt['customer_age'] = df_opt['customer_age'].astype('float32')
df_opt['unit_price'] = df_opt['unit_price'].astype('float32')
df_opt['total_value'] = df_opt['total_value'].astype('float32')
print(f'Memory before: {df.memory_usage(deep=True).sum() / 1024:.1f} KB')
print(f'Memory after: {df_opt.memory_usage(deep=True).sum() / 1024:.1f} KB')

Memory before: 571.9 KB
Memory after: 174.8 KB


### Save and commit

In [6]:
df_opt.to_csv('../data/processed/sales_clean.csv', index=False)